In [12]:
import os
os.environ["VISIBLE_DEVICES"] = "0"
import onnxruntime as ort
import torchaudio
import torch
import numpy as np
import whisper

In [13]:
# ONNXロード
ONNX_PATH = "./speech_tokenizer_v2.onnx"
option = ort.SessionOptions()
option.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
option.intra_op_num_threads = 1
providers = ["CUDAExecutionProvider"]
ort_session = ort.InferenceSession(ONNX_PATH, sess_options=option, providers=providers)

for i in ort_session.get_inputs():
    print(i.name, i.shape, i.type)

feats [1, 128, 'T'] tensor(float)
feats_length [1] tensor(int32)


2026-05-18 20:49:58.124247342 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 8 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-05-18 20:49:58.128140310 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-05-18 20:49:58.128150475 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [14]:
# 音声ロード
SPEECH_PATH = "/db/Coco-Nut/Coco-Nut/wav/0/0000.wav"
wav, sr = torchaudio.load(SPEECH_PATH)
print(wav.shape, sr)  # [2, T], 44100

# 16kHzにリサンプル
if sr != 16000:
    wav = torchaudio.functional.resample(wav, sr, 16000)
# モノラル化
wav = wav.mean(dim=0, keepdim=True)  # [1, T]

torch.Size([2, 99666]) 44100


In [16]:
# log-mel spectrogram作成
feat = whisper.log_mel_spectrogram(wav, n_mels=128)      # [1, 128, T]

In [19]:
# 推論
# for i in session.get_inputs():
#     print("Input:", i.name, i.shape, i.type)

speech_token = ort_session.run(None, {ort_session.get_inputs()[0].name: feat.detach().cpu().numpy(),
                                              ort_session.get_inputs()[1].name: np.array([feat.shape[2]], dtype=np.int32)})[0].flatten().tolist()


print(speech_token)

[1708, 4174, 4731, 4515, 4515, 2256, 5426, 4688, 2303, 2273, 2864, 5087, 4843, 4218, 6429, 4504, 3667, 6179, 3586, 2112, 1029, 2214, 2240, 4523, 4616, 5345, 1821, 3423, 1167, 3216, 126, 641, 1336, 1163, 224, 647, 1258, 1983, 1956, 2037, 2058, 2241, 2189, 2309, 4967, 3184, 1564, 749, 740, 2927, 5202, 6177, 5938, 1038, 65, 272, 1216]
